In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
project_root = "/root/owlnet"
sys.path.append(project_root)
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"
os.chdir(project_root)
print(os.getcwd())


In [ ]:
from pathlib import Path

import IPython.display as ipd

from owlnet.core.utils import load_config, chop_file
from owlnet.core.utils import display_audio_file

config = load_config("settings/config.json")
file_to_use = Path(config["data_dir"]) / f"{config['test_file']}"

# "test_file": "nest1_audiomoth_stokewake_2025/20250708/248D9B045CC5EE36_20250708_000010.WAV",
# "test_file": "nest3_smu2_holtlodge_2025/2MM08624_20250628_023002.wav",

display_audio_file(config, file_to_use)


In [ ]:
from owlnet.core.utils import infer_abs_unix_timestamp


start = infer_abs_unix_timestamp(str(file_to_use.stem))
chunks, chunks_crossing_times, _ = chop_file(config, file_to_use, t_init=start, display=True)
# ipd.Audio(Path(config["data_dir"]) / config["test_file"])


In [ ]:
import matplotlib.pyplot as plt
from owlnet.core.utils import display_datetime

disp_chunk = 8
for i, (chunk, times) in enumerate(zip(chunks, chunks_crossing_times)):
    if i == disp_chunk:
        plt.imshow(chunk.squeeze(), aspect=0.15)
        print(f"start_time={display_datetime(round(times[0]))}\nend_time={display_datetime(round(times[1]))}")
        break

In [ ]:
import json
import torch

filename = Path(config['test_file'])
save_dir = Path(config['data_dir']).resolve() / filename.with_suffix("")
print(f"Saving to: {save_dir}")
if not save_dir.exists():
    save_dir.mkdir()

config["ext"] = filename.suffix
with open(save_dir / "config.json", "w") as fh:
    json.dump(config, fh)

    # for i, chunk in enumerate(chunks):
    #     torch.save(chunk, save_dir / f"{filename}_{i}.pth")
